In [1]:
from datetime import datetime, time, timedelta
from Database.TPData import TPData
from Database.DB_reader import Database
from OrderBook.OrderBook import OrderBookSnaps
import os
import itertools
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
def instrument_key(market, tenor, venue):
    data_class = TPData()
    instid = data_class.instid_dict[market][venue]
    prodid = data_class.seqid_dict(data_class.com_dict[market])[tenor.upper()]
    return str(instid) + '_' + str(prodid)

def itemid(t, p_d):
    if p_d is None:
        return 0
    else:
        data_class = TPData()
        return data_class.calc_itemid(t, p_d)

def get_dataframe(ob_class, m, t, v, pd1, pd2):
    data_class = TPData()
    df_exp = pd.concat([ob_class.best_bid_all, ob_class.best_ask_all,
                        ob_class.best_bid_any_noi, ob_class.best_ask_any_noi], axis=1)
    df_exp.index.name = 'datetime'
    df_exp.columns = ['bidbestprice', 'askbestprice', 'bidbestprice_aonn', 'askbestprice_aonn']
    df_exp = data_class.process_best_orders_all(df_exp)
    df_exp['instkey'] = instrument_key(m, t, v)
    df_exp['firstsequenceitemid'] = itemid(t, pd1)
    df_exp['secondsequenceitemid'] = itemid(t, pd2)
    current_timestamp = datetime.now()
    df_exp['upload_timestamp'] = current_timestamp
    df_exp = df_exp.reset_index()
    return df_exp[['datetime', 'instkey', 'firstsequenceitemid', 'secondsequenceitemid',
                   'bidbestprice', 'askbestprice', 'upload_timestamp', 'bidbestprice_aonn', 'askbestprice_aonn']]

def save_dataframe(df_exp):
    conn = Database()
    conn._connect()
    df_exp.to_sql('ba_price', conn.engine, schema='best_orders', if_exists='append', index=False)
    
    
def export_order_book(mkt, tenor, prod, venue_list, prod_date, date,
                      default_path):
    start_time = time(8, 0, 0)
    end_time = time(18, 0, 0)


    def load_ob(m, t, dt, p_d, bT, eT):
        ob_class = OrderBookSnaps(verbose=True)
        file_path = l_path + t.split('_')[0] + '/'
        file_name = m + '_' + t.split('_')[0] + '_' + p_d.strftime('%y%m%d') + '_' + dt.strftime('%y%m%d') + '.p'
        print('%s Loading OrderBook %d...' % (dt.strftime('%y-%m-%d'), 0))
        time_load = ob_class.import_data(file_path + file_name)
        print('OrderBook %d created in %d sec' % (0, time_load))
        # ob_class.LoB_truncate(thres_vol=1)
        return ob_class.LoB_select(bT, eT, freq=None)
    
    data_class = TPData()
    data_class.create_connection('PostgreSQL', path_name='z:\EnergyTrading\configDB.json')

    bT = datetime.combine(date, start_time)
    eT = datetime.combine(date, end_time)
    
    

    ob_class = OrderBookSnaps(verbose=True)
    ob_class.clear()
    LoB_dict, time_list = load_ob(m, t, bT, prod_date, bT, eT)
    ob_class.update_data(LoB_dict, time_list)

    # Save bid asks
    venue = venue_list[0]
    df_exp = get_dataframe(ob_class, mkt, tenor, venue, prod_date, None)
    save_dataframe(df_exp)
    ob_class.clear()
    return 

In [3]:
# manual_bool = True
# if manual_bool:
#     start_date = datetime(2025, 2, 20)
#     end_date = datetime(2025, 2, 20)
# else:
#     prev_wday = previous_working_day(datetime.today().date())
#     start_date = datetime.combine(prev_wday, time(0, 0))
#     end_date = datetime.combine(prev_wday, time(0, 0))
#     del prev_wday

# ########DE part
# default_path = '//192.168.10.91/data/Data/orderbooks/'
# l_path = '//192.168.10.91/data/Data/orderbooks/base/'
# mkt_list = ['de']
# tenor_list = ['m']
# prod = 'base'
# venue_list = ['eex']
# tn_dict = {
#            'm': [1, 2]
           
#            }


# n_s = 2
# dates = pd.date_range(start_date, end_date, freq='B')

# combinations = []
# for mkt in mkt_list:
#     for tenor in tenor_list:
#         for tn in tn_dict[tenor]:
#             combinations.append((mkt, tenor, tn))

# mkt_list_ = [x[0] for x in combinations]
# tenor_list_ = [x[1] for x in combinations]
# tn_list_ = [x[2] for x in combinations]

# product_date = [dates.shift(1, freq='B') if t == 'da' else
#                 dates.shift(1, freq='D') if t == 'd' else
#                 dates.shift(tn, freq='W-MON') if t == 'w' else
#                 (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
#                 (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
#                 (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
#                 for t, tn in zip(tenor_list_, tn_list_)]

# results=[]
# for m, t, prod_d in zip(mkt_list_, tenor_list_, product_date):
#     for dt, p_d in zip(dates, prod_d):
#         print((m, t, prod, venue_list, p_d, dt))
#         try:
#             err_dict = export_order_book(m, t, prod, venue_list, p_d, dt,
#                                      default_path)
#             print('OK')

#         except Exception as err:
#             print(f'Failed: {err}')

In [4]:
manual_bool = True
if manual_bool:
    start_date = datetime(2025, 2, 25)
    end_date = datetime(2025, 2, 25)
else:
    prev_wday = previous_working_day(datetime.today().date())
    start_date = datetime.combine(prev_wday, time(0, 0))
    end_date = datetime.combine(prev_wday, time(0, 0))
    del prev_wday

########DE part
default_path = '//192.168.10.91/data/Data/orderbooks/'
l_path = '//192.168.10.91/data/Data/orderbooks/base/'
mkt_list = ['the']
tenor_list = ['da', 'm']
prod = 'base'
venue_list = ['eex']
tn_dict = {'da': [1],
           'm': [1]}

n_s = 2
dates = pd.date_range(start_date, end_date, freq='B')

combinations = []
for mkt in mkt_list:
    for tenor in tenor_list:
        for tn in tn_dict[tenor]:
            combinations.append((mkt, tenor, tn))

mkt_list_ = [x[0] for x in combinations]
tenor_list_ = [x[1] for x in combinations]
tn_list_ = [x[2] for x in combinations]

product_date = [dates.shift(1, freq='B') if t == 'da' else
                dates.shift(1, freq='D') if t == 'd' else
                dates.shift(tn, freq='W-MON') if t == 'w' else
                (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                for t, tn in zip(tenor_list_, tn_list_)]

results=[]
for m, t, prod_d in zip(mkt_list_, tenor_list_, product_date):
    for dt, p_d in zip(dates, prod_d):
        print((m, t, prod, venue_list, p_d, dt))
        try:
            err_dict = export_order_book(m, t, prod, venue_list, p_d, dt,
                                     default_path)
            print('OK')

        except Exception as err:
            print(f'Failed: {err}')
            raise

('the', 'da', 'base', ['eex'], Timestamp('2025-02-26 00:00:00', freq='B'), Timestamp('2025-02-25 00:00:00', freq='B'))
25-02-25 Loading OrderBook 0...
Failed: _unpickle_timestamp() takes exactly 3 positional arguments (4 given)


TypeError: _unpickle_timestamp() takes exactly 3 positional arguments (4 given)